In [ ]:
"""
   Installing openlside package for reading WSI file
"""
!pip install openslide-python==1.4.1

In [ ]:
"""
  Installing other dependencies for openslide
"""
!apt-get update -qq
!apt-get install -y openslide-tools

In [ ]:
"""
  Installing package that can be used for Otsu tissue segmentation
"""
!pip install tiatoolbox

In [ ]:
"""
  Command for mounting the drive.
"""
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
   Changing the directory to the required working directory. Please change the
   path accordingly where this source file save on the Google drive.
"""
%cd '/content/drive/My Drive/2_PHDWork/1_Objective-I/4_Enlargement_Strategy/1_ComparisonStudy/'

In [ ]:
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
import time
import cv2 as cv
from scipy.spatial.distance import directed_hausdorff
from PIL import Image

from openslide import OpenSlide
from tiatoolbox.tools.tissuemask import OtsuTissueMasker
from shapely import get_coordinates, concave_hull, buffer, LineString, Point
from shapely.geometry import Polygon
from shapely.affinity import translate
from shapely.ops import unary_union
from shapely.geometry.multipolygon import MultiPolygon

In [ ]:
"""
  In PANDA dataset the thumbnail mask is already available. Make this variable
  true if you want the proposed method to work with otsu based thumbnail mask
  for PANDA. Please note it is applicable only for PANDA dataset.
"""
ENFORCE_OTSU_THUMBNAIL_MASK_FOR_PANDA = False

In [ ]:
"""
Writes the comparison data to a csv file.

Parameters:
     log_data: the comparison data to be written.

Returns:
     None
"""
def log_comparison_data(log_data):
  fields = ['dataset','f_name','t_row','t_col','i_row','i_col','read_time',
            's_cntr_trans_time','s_msk_fill_time','m_cntr_trans_time',
            'm_msk_fill_time','org_msk_gen_time','s_mem_req','m_mem_req',
            'o_mem_req','s_accuracy','s_dice','s_iou','s_hdit','s_tpr','s_fpr',
            'm_accuracy','m_dice','m_iou','m_hdit','m_tpr','m_fpr']

  """
    The path and name of the log file to which comparison data is writted.
    Appropriately set the name to get the desired log file.

    Note: Set it to appropriate filepath and set the 'logdata' to True for
      writing comparison data.
  """
  log_file_name = './log_files/proposed_method_results.csv'

  # Creating a log file if it does not exist.
  if(not os.path.isfile(log_file_name)):
    with open(log_file_name, 'w') as csvfile:
      # creating a csv writer object
      csvwriter = csv.writer(csvfile)
      # writing the fields
      csvwriter.writerow(fields)

  # Writing comparison data to the file.
  try:
    with open(log_file_name,"a") as csvFile:
      writer = csv.DictWriter(csvFile, fieldnames=fields)
      writer.writerow(log_data)
  except IOError:
    print("Writing of Training data failed")

  return

In [ ]:
"""
  Method to generate the thumbnail mask of the given WSI and construct the
  retrieve the boundary points of the mask boundary.

  Parameters:
      wsiname     : the name of the WSI which contains the thumbnail image
      log_data    : the data to be written to the log file.
      thm_msk_file: the thumbnail mask file if the thumbnail mask is already
                    available
      level       : the WSI level to be processed.

  Returns:
      o_contours  : the thumbnail mask boundary points
      t_size      : the thumbnail size
      l_size      : the required mask size
      log_data    : the data to be logged
"""
def get_thumbnail_mask_contour(wsiname,log_data,thm_msk_file=None,level=0):
  wsi = OpenSlide(wsiname)

  # Populating the details of the WSI
  t_level =  wsi.level_count-1
  l_size = (wsi.level_dimensions[level][1],wsi.level_dimensions[level][0])
  t_size = (wsi.level_dimensions[t_level][1],wsi.level_dimensions[t_level][0])
  log_data['t_row'] = t_size[0] # Thumbnail Size
  log_data['t_col'] = t_size[1] # Thumbnail Size
  log_data['i_row'] = l_size[0] # Required Mask Size
  log_data['i_col'] = l_size[1] # required Mask Size

  if(thm_msk_file is None):
    start_time = time.process_time()
    t_image = np.array(wsi.read_region((0,0),t_level,wsi.level_dimensions[t_level]))
    t_image = cv.cvtColor(t_image,cv.COLOR_RGBA2RGB)
    i_size = t_image.shape
    t_image = np.reshape(t_image,(1,i_size[0],i_size[1],i_size[2]))
    masker = OtsuTissueMasker()
    masks = masker.fit_transform(t_image) # Here we get a list of gray scale image
    true_index = np.asarray(masks[0] == True).nonzero()
    t_mask = np.zeros(masks[0].shape,dtype=np.uint8)
    t_mask[true_index] = 255
    o_contours, _ = cv.findContours(t_mask,cv.RETR_EXTERNAL,
                                              cv.CHAIN_APPROX_NONE)
    end_time = time.process_time()
    read_time = (end_time - start_time)
  else:
    if(ENFORCE_OTSU_THUMBNAIL_MASK_FOR_PANDA):
      # Use Otsu based thumbnail mask for PANDA.
      start_time = time.process_time()
      t_image = np.array(wsi.read_region((0,0),t_level,wsi.level_dimensions[t_level]))
      t_image = cv.cvtColor(t_image,cv.COLOR_RGBA2RGB)
      i_size = t_image.shape
      t_image = np.reshape(t_image,(1,i_size[0],i_size[1],i_size[2]))
      masker = OtsuTissueMasker()
      masks = masker.fit_transform(t_image) # Here we get a list of gray scale image
      true_index = np.asarray(masks[0] == True).nonzero()
      t_mask = np.zeros(masks[0].shape,dtype=np.uint8)
      t_mask[true_index] = 255
      o_contours, _ = cv.findContours(t_mask,cv.RETR_EXTERNAL,
                                              cv.CHAIN_APPROX_NONE)
      end_time = time.process_time()
      read_time = (end_time - start_time)
    else:
      # Use thumbnail mask provided by the PANDA dataset.
      mask_wsi  = OpenSlide(thm_msk_file)
      t_mask = np.zeros((t_size[0],t_size[1]),dtype=np.uint8)
      start_time = time.process_time()
      f_mask = np.array(mask_wsi.read_region((0,0),t_level,mask_wsi.level_dimensions[t_level]))
      f_mask = (cv.cvtColor(f_mask,cv.COLOR_RGBA2RGB))[:,:,0]
      t_mask[f_mask > 0] = 255
      o_contours, _ = cv.findContours(t_mask,cv.RETR_EXTERNAL,
                                              cv.CHAIN_APPROX_NONE)
      end_time = time.process_time()
      read_time = (end_time - start_time)
      mask_wsi.close()
      del mask_wsi

  log_data['read_time'] = read_time

  return o_contours, t_size, l_size, log_data


"""
  Method to retrieve the ground-truth mask for performance analysis.

  Parameters:
        maskfile : the ground-truth mask file
        datst    : the dataset name
        log_data : the data to be logged
        level    : the WSI level to be processed.

  Returns:
       org_mask : the ground-truth mask
       log_data : the data to be logged.
"""
def get_ground_truth_mask(maskfile,datst,log_data,level=0):
  if(datst == "PANDA"):
    mask_wsi  = OpenSlide(maskfile)
    start_time = time.process_time()
    full_mask = np.array(mask_wsi.read_region((0,0),level,
                                              mask_wsi.level_dimensions[level]))
    full_mask = (cv.cvtColor(full_mask,cv.COLOR_RGBA2RGB))[:,:,0]
    org_mask = np.zeros((full_mask.shape[0],full_mask.shape[1]),dtype=np.uint8)
    org_mask[full_mask > 0] = 255
    end_time = time.process_time()
    org_msk_gen_time = (end_time - start_time)

    """
      Note: Changes can be incorpoarted here if some other method mask such as
            the Otsu based are required.
    """

  else:
    # Load the ground truth mask
    start_time = time.process_time()
    org_mask = cv.imread(maskfile, cv.IMREAD_GRAYSCALE)
    org_mask = (org_mask > 0).astype(np.uint8)
    org_mask *= 255
    end_time = time.process_time()
    org_msk_gen_time = (end_time - start_time)

  log_data['org_msk_gen_time'] = org_msk_gen_time

  """
    Use only if original contour points are required.
  """
  # o_cntrs, _ = cv.findContours(org_mask,cv.RETR_EXTERNAL,cv.CHAIN_APPROX_NONE)
  return org_mask, log_data

In [ ]:
"""
  Method that scales the thumbnail contour points based on the scaling factor.
  This method performs the simple scaling operation.

  Parameters:
        contours : the thumbnail mask contour points
        t_size   : the thumbnail size
        l_size   : the required mask size
        log_data : the data to be logged

  Returns:
        bin_mask : the generated mask
        log_data : the data to be logged
        new_points: the scaled contour points
"""
def get_equation_based_tissue_mask(contours,t_size,l_size,log_data):
  s_r = l_size[0] / t_size[0]
  s_c = l_size[1] / t_size[1]
  bin_mask = np.zeros(l_size,dtype=np.uint8)
  new_points = []

  # Computing on the approximated contours
  start_time = time.process_time()
  for pnts in contours:
    wrk_pnts = pnts.astype(np.float32)
    wrk_pnts[:,:,0] = wrk_pnts[:,:,0] * s_c
    wrk_pnts[:,:,1] = wrk_pnts[:,:,1] * s_r
    wrk_pnts = (np.floor(wrk_pnts)).astype(np.int32)
    new_points.append(wrk_pnts)
  end_time = time.process_time()
  cntr_trans_time = (end_time - start_time)
  log_data['s_cntr_trans_time'] = cntr_trans_time

  start_time = time.process_time()
  bin_mask = cv.fillPoly(bin_mask, new_points, (255))
  end_time = time.process_time()
  msk_fill_time = (end_time - start_time)
  log_data['s_msk_fill_time'] = msk_fill_time

  mem_req = 0
  for pnts in contours:
    total_points = 0
    for i in range(pnts.shape[0]):
      for j in range(pnts.shape[1]):
        total_points += 2
    mem_req +=  total_points * 4

  log_data['s_mem_req'] = mem_req

  # plt.imshow(bin_mask,cmap='gray')
  # plt.show()
  return bin_mask,log_data,new_points

In [ ]:
"""
  Method that performs the mask generation using the steps mentioned in the
  proposed method.

  Parameters:
        o_contours  : the thumbnail mask contour points
        t_size      : the thumbnail size
        l_size      : the required mask size
        log_data    : the data to be logged

  Returns:
        bin_mask    : the generated mask
"""
def proposed_method(o_contours,t_size,l_size,log_data):
  par = 0.5  # The shift parameter. Change the value appropriately for ablation test.
  s_r = l_size[0] / t_size[0]
  s_c = l_size[1] / t_size[1]
  """
    Use only when you want to generate the output image
  """
  # i_size = (l_size[0],l_size[1],3)
  # output_img = np.zeros(i_size,dtype=np.uint8)
  bin_mask = np.zeros(l_size,dtype=np.uint8)
  new_points = []
  o_s_points = []

  # Step-1: Transformation of co-ordinates.
  start_time = time.process_time()
  for i in range(len(o_contours)):
    cnt = np.reshape(o_contours[i].copy(),(-1,2)).astype(np.float32)

    if(cnt.shape[0] <= 10):
      if(cnt.shape[0] == 1):
        poly1 = buffer(Point(cnt), 0.3,join_style='mitre')
      else:
        poly1 = buffer(LineString(cnt.tolist()), 0.3,join_style='mitre')

      comb_ply = scale_polygon_coordinates(poly1,s_r,s_c)
      if not comb_ply.is_valid:
        comb_ply = comb_ply.buffer(0.1,join_style='mitre')
    else:
      poly1 = Polygon(cnt)
      poly2 = translate(poly1, xoff=par, yoff=par)    # Bottom-right
      poly3 = translate(poly1, xoff=-par, yoff=par)   # Bottom-left
      poly4 = translate(poly1, xoff=par, yoff=-par)   # Top-right

      poly1 = scale_polygon_coordinates(poly1,s_r,s_c)
      poly2 = scale_polygon_coordinates(poly2,s_r,s_c)
      poly3 = scale_polygon_coordinates(poly3,s_r,s_c)
      poly4 = scale_polygon_coordinates(poly4,s_r,s_c)

      if not poly1.is_valid:
        poly1 = poly1.buffer(0.1,join_style='mitre')
      if not poly2.is_valid:
        poly2 = poly2.buffer(0.1,join_style='mitre')
      if not poly3.is_valid:
        poly3 = poly3.buffer(0.1,join_style='mitre')
      if not poly4.is_valid:
        poly4 = poly4.buffer(0.1,join_style='mitre')

      comb_ply = unary_union([poly1,poly2,poly3,poly4])

      if isinstance(comb_ply, MultiPolygon):
        comb_ply = concave_hull(comb_ply,ratio=0.001)

    new_points.append(np.reshape(np.array(list(comb_ply.exterior.coords)),(-1,1,2)))
    o_s_points.append(np.reshape(np.array(list(poly1.exterior.coords)),(-1,1,2)))

  end_time = time.process_time()
  cntr_trans_time = (end_time - start_time)
  log_data['m_cntr_trans_time'] = cntr_trans_time

  o_s_points = convert_to_displayable_contours(o_s_points,l_size[0],l_size[1])
  new_points = convert_to_displayable_contours(new_points,l_size[0],l_size[1])

  # Step-2: Generating the mask.
  start_time = time.process_time()
  bin_mask = cv.fillPoly(bin_mask, new_points, (255))
  end_time = time.process_time()
  msk_fill_time = (end_time - start_time)
  log_data['m_msk_fill_time'] = msk_fill_time

  r_mem_req = 0
  for pnts in new_points:
    total_points = 0
    for i in range(pnts.shape[0]):
      for j in range(pnts.shape[1]):
        total_points += 2
    r_mem_req +=  total_points * 4
  log_data['m_mem_req'] = r_mem_req
  log_data['o_mem_req'] = l_size[0] * l_size[1] * 3

  """
    Use only when you want to generate the output image
  """
  # cv.drawContours(output_img, o_s_points, -1, (255,0,0), 1)
  # cv.drawContours(output_img, new_points, -1, (0,255,0), 1)

  return bin_mask,log_data,new_points

"""
   Auxilary method used to scale the co-ordinate points

   Parameters:
        ploygon : the polygon points to be scaled.
        s_r     : the scaling factor in row direction
        s_c     : the scaling factor in column direction

   Returns:
        s_polygon : the scaled polygon
"""
def scale_polygon_coordinates(polygon,s_r,s_c):
  coordinates = get_coordinates(polygon)
  coordinates[:,0] *= s_c
  coordinates[:,1] *= s_r

  s_polygon = Polygon(coordinates)

  return s_polygon

"""
  Method to convert the float type co-ordinates to int so that it can be mapped
  to image pixels.

  Parameters:
        contours  : the co-ordinates to be converted to int.
        max_r     : the maximum row value.
        max_c     : the maximum column value.

  Returns:
        dis_cont  : the converted co-ordinates.
"""
def convert_to_displayable_contours(contours,max_r,max_c):
  dis_cont = []
  for i in range(len(contours)):
    cnt = np.ceil(contours[i].copy()).astype(np.int32)
    cnt[:,:,0] = np.clip(cnt[:,:,0],0,max_c)
    cnt[:,:,1] = np.clip(cnt[:,:,1],0,max_r)
    dis_cont.append(cnt)

  return dis_cont

In [ ]:
"""
Performs a pixel to pixel matching between the given masks and returns the
accuracy, Dice's and Jaccard's coefficient.

Parameters:
     org:     the original labels.
     pred:    the predicted labels.

Returns:
     accuracy:    the percentage of pixels that matches.
     dice:        the Dice's coefficients for tissue pixels.
     iou:         the Intersection over Union for tissue pixels.
     hausdorff_distance: the housdroff distance for tissue pixels.
     TPR:         the true positive rate.
     FPR:         the false positive rate
"""
def measure_performance_metrics(org,pred):
  # Converting the values into 0 and 1.
  i1 = org // 255
  i2 = pred // 255

  # Calculating the TP, TN, FP and FN
  TP = np.sum((i1 * i2))
  TN = np.sum((1 - i1) * (1 - i2))
  FP = np.sum((((i2 - i1) + 1)//2))
  FN = np.sum((((i1 - i2) + 1)//2))

  accuracy = round(((TP + TN)/(TP + TN + FP + FN)),6) * 100

  area_intersection = TP
  area_union = TP + FN + FP

  if(area_union != 0):
    dice = 2 * area_intersection / (np.sum(i1) + np.sum(i2))
    iou = area_intersection / area_union
    # jaccard = area_intersection / area_union
  else:
    dice = 0
    iou = 0

  # Find contours (boundary points) of the masks
  contours1, _ = cv.findContours(org, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_NONE)
  contours2, _ = cv.findContours(pred, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_NONE)

  # Extract boundary points
  points1 = np.vstack(contours1).squeeze()
  points2 = np.vstack(contours2).squeeze()

  u = directed_hausdorff(points1, points2)[0]
  v = directed_hausdorff(points2, points1)[0]
  hausdorff_distance = max(u, v)
  # hausdorff_distance = 0

  TPR = round((TP / (TP + FN)),8)
  FPR = round((FP / (FP + TN)),8)

  return (accuracy,dice,iou,hausdorff_distance,TPR,FPR)

"""
  Method for saving the predicted mask.

  Parameters:
      pred_mask: the predicted mask.
      output_name: the name of the output file.

  Returns:
      None
"""
def save_mask(pred_mask,output_name):
  mask_img = Image.fromarray(pred_mask.astype("uint8"))

  # Save to file
  mask_img.save(output_name)

  return

In [ ]:
"""
  The main method where the execution starts.

  Parameters:
       logdata: if true log file is created with all the logs
       savemask: if true the predicted mask is saved.
  Returns: None
"""
def main(logdata=False,savemask=False):
  # List of dataset from which WSIs are considered.
  dataset = ["CPTAC-CM","CPTAC-UCEC","PANDA"]

  # The WSI level to be processed
  level = 0

  # The path location for each dataset.
  dataset_details = {
      "CPTAC-CM" : {"wsi_location"  : "../../0_Dataset/1_Dataset/0_Pure/1_CPTAC-CM/",       # Location where the WSIs are saved.
                    "grdtru_msk_loc": "../../0_Dataset/1_Dataset/2_Masks/1_CPTAC-CM/PNG/",  # Location where the ground truth masks are saved.
                    "thumbnail_msk" : "",                                                   # Location where the thumbnail masks are saved.
                    "output_loc"    : "./0_Generated_Mask/proposed_method/1_CPTAC-CM/"     # Location where the predicted masks are to be saved.
                   },
      "CPTAC-UCEC" : {"wsi_location"  : "../../0_Dataset/1_Dataset/0_Pure/4_CPTAC-UCEC/",
                      "grdtru_msk_loc": "../../0_Dataset/1_Dataset/2_Masks/4_CPTAC-UCEC/PNG/",
                      "thumbnail_msk" : "",
                      "output_loc"    : "./0_Generated_Mask/proposed_method/4_CPTAC-UCEC/"
                     },
      "PANDA" : {"wsi_location"       : "../../../2_Objective-II/0_Dataset/1_Panda/train_images/radboud/",
                      "grdtru_msk_loc": "../../../2_Objective-II/0_Dataset/1_Panda/train_label_masks/radboud/",
                      "thumbnail_msk" : "../../../2_Objective-II/0_Dataset/1_Panda/train_label_masks/radboud/",
                      "output_loc"    : "./0_Generated_Mask/proposed_method/5_PANDA/1_dataset_based/" # location for masks generated from dataset provided thumbnail mask
                      #"output_loc"    : "./0_Generated_Mask/proposed_method/5_PANDA/2_Otsu_based/"  # location for masks generated from otsu thumbnail mask
                     }
      }
  ind = 0
  for datst in dataset:
    ind += 1
    """
      Code for avoiding the execution of selected dataset.
    """
    # if(ind != 3):
    #   continue
    print(f'Processing dataset: {datst}')
    wsi_location = dataset_details[datst]["wsi_location"]
    grdtru_msk_loc = dataset_details[datst]["grdtru_msk_loc"]
    output_loc = dataset_details[datst]["output_loc"]
    if(datst == "PANDA"):
      thumb_mks_path = dataset_details[datst]["thumbnail_msk"]
    else:
      thumb_mks_path = None

    if(os.path.isdir(wsi_location)):
      files_names = os.listdir(wsi_location)
      print(f'\tProcessing a total of {len(files_names)} WSI files.')

      index = 0
      limit = len(files_names)
      while(index < limit):
        files = files_names[index]
        m_name, ext = os.path.splitext(files)
        wsiname = wsi_location + files
        if(datst == "PANDA"):
          maskfile = grdtru_msk_loc + m_name + "_mask" + ext
          thm_msk_file = grdtru_msk_loc + m_name + "_mask" + ext
        else:
          maskfile = grdtru_msk_loc + m_name + "_mask" + ".png"
          thm_msk_file = None

        output_name = output_loc + m_name + "_predicted_mask" + ".png"
        log_data = {}
        log_data['dataset'] = datst
        log_data['f_name'] = files

        print(f'\t\t{index + 1} Processing WSI file: {files}')
        o_contours, t_size, l_size, log_data = get_thumbnail_mask_contour(wsiname,log_data,thm_msk_file,level) # obtaining the thumbnail mask co-ordinates
        s_mask,log_data,o_s_points = get_equation_based_tissue_mask(o_contours,t_size,l_size,log_data) # performing simple scaling
        mask,log_data,n_cntrs = proposed_method(o_contours,t_size,l_size,log_data) # generating mask as per the proposed method

        org_mask, log_data = get_ground_truth_mask(maskfile,datst,log_data,level) # generating the ground-truth mask for comparison

        # Comparison with the contours and mask generated by simple scaling startegy
        s_accuracy,s_dice,s_iou,s_hausdorff_distance,s_tpr,s_fpr = measure_performance_metrics(org_mask,s_mask)

        # Comparison with the contours and mask generated by new startegy
        accuracy,dice,iou,hausdorff_distance,tpr,fpr = measure_performance_metrics(org_mask,mask)

        log_data['s_accuracy'] = s_accuracy
        log_data['s_dice']     = s_dice
        log_data['s_iou']      = s_iou
        log_data['s_hdit']     = s_hausdorff_distance
        log_data['s_tpr']      = s_tpr
        log_data['s_fpr']      = s_fpr

        log_data['m_accuracy'] = accuracy
        log_data['m_dice']     = dice
        log_data['m_iou']      = iou
        log_data['m_hdit']     = hausdorff_distance
        log_data['m_tpr']      = tpr
        log_data['m_fpr']      = fpr

        if(savemask):
          save_mask(mask,output_name)
        if(logdata):
          log_comparison_data(log_data)
        # break
        index += 1

        del o_contours, t_size, l_size, log_data,s_mask,o_s_points,mask,n_cntrs
        del org_mask,accuracy,dice,iou,hausdorff_distance,tpr,fpr,s_accuracy,s_dice,s_iou,s_hausdorff_distance,s_tpr,s_fpr

        # break

  return

In [ ]:
main() #Execution starts here